# Shopping Dataset — Python & Pandas Basics: Data Exploration and Cleaning

**Objective:** Learn Python basics and perform basic data exploration and cleaning using Pandas.

**Dataset:** Shopping / e-commerce order data (`data/raw/shopping_dataset.csv`), modeled on the
structure of the [Kaggle "Shopping Dataset"](https://www.kaggle.com/datasets/anvitkumar/shopping-dataset).

> **Note:** This notebook ships with a synthetic sample dataset that has the same column
> structure and the same kinds of issues (missing values, duplicate rows) you'd find in the
> real Kaggle file. To use the real data instead, download the CSV from Kaggle, place it at
> `data/raw/shopping_dataset.csv` (replacing the sample), and re-run all cells. If your real
> file uses different column names, update the `PRICE_COL` / `QTY_COL` variables in Step 6.

**Steps covered:**
1. Load the CSV into a DataFrame
2. Explore the data (head/tail, shape, columns, dtypes)
3. Handle missing values (identify, fill/drop)
4. Basic operations (filter rows, select columns)
5. Remove duplicates
6. Create a derived column (`total_amount = price * quantity`)
7. Save the cleaned dataset as a new CSV


## Step 0: Import libraries

We only need `pandas` for this exercise. `numpy` is imported too since it's commonly used alongside pandas.

In [7]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
print("Pandas version:", pd.__version__)


Pandas version: 2.3.3


## Step 1: Load the CSV dataset into a Pandas DataFrame

In [8]:
RAW_PATH = "../data/raw/shopping_dataset.csv"

df = pd.read_csv(RAW_PATH)
print(f"Loaded {len(df)} rows from {RAW_PATH}")


Loaded 260 rows from ../data/raw/shopping_dataset.csv


## Step 2: Explore the data

Before cleaning anything, it's good practice to get a feel for the dataset: what columns it
has, what type of data each column holds, and how big it is.


In [14]:
# First 5 rows
df.head()


,order_id,customer_name,product,category,price,quantity,order_date,payment_method,region
0,1103,Priya Sharma,Wireless Mouse,Electronics,211.29,5.0,2024-04-12,UPI,South
1,1246,Pooja Kapoor,Action Figure,Toys,120.14,5.0,2024-12-07,UPI,North
2,1195,Rohan Verma,Denim Jacket,Clothing,12.36,2.0,2024-11-30,Credit Card,South
3,1118,Vikram Patel,Resistance Bands,Sports,48.49,1.0,2024-03-10,Debit Card,North
4,1191,Karan Iyer,Shampoo,Beauty,65.86,2.0,2024-02-28,Credit Card,West


In [10]:
# Last 5 rows
df.tail()


,order_id,customer_name,product,category,price,quantity,order_date,payment_method,region
255,1138,Ritu Mehta,Cookbook,Books,107.99,5.0,2024-10-01,UPI,East
256,1122,Rohan Reddy,Resistance Bands,Sports,NaN,1.0,2024-09-24,Net Banking,South
257,1073,Rohan Singh,Earbuds,Electronics,34.33,4.0,2024-02-02,UPI,East
258,1236,Neha Gupta,Sunscreen SPF50,Beauty,118.85,2.0,2024-03-21,Cash on Delivery,North
259,1038,Priya Singh,Storage Box,Home & Kitchen,96.55,5.0,2024-09-29,Debit Card,South


In [11]:
# Shape: (rows, columns)
print("Shape:", df.shape)


Shape: (260, 9)


In [12]:
# Column names
print("Columns:", list(df.columns))


Columns: ['order_id', 'customer_name', 'product', 'category', 'price', 'quantity', 'order_date', 'payment_method', 'region']


In [13]:
# Data types of each column
df.dtypes


order_id            int64
customer_name      object
product            object
category           object
price             float64
quantity          float64
order_date         object
payment_method     object
region             object
dtype: object

In [8]:
# A quick statistical summary of numeric columns
df.describe()


,order_id,price,quantity
count,260.000000,254.000000,256.000000
mean,1125.565385,131.992087,3.093750
std,71.989539,68.734173,1.319165
min,1001.000000,6.100000,1.000000
25%,1064.750000,76.645000,2.000000
50%,1124.500000,125.810000,3.000000
75%,1187.250000,192.232500,4.000000
max,1250.000000,245.150000,5.000000


In [9]:
# A concise summary: dtypes, non-null counts, memory usage
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        260 non-null    int64  
 1   customer_name   260 non-null    str    
 2   product         260 non-null    str    
 3   category        248 non-null    str    
 4   price           254 non-null    float64
 5   quantity        256 non-null    float64
 6   order_date      260 non-null    str    
 7   payment_method  251 non-null    str    
 8   region          260 non-null    str    
dtypes: float64(2), int64(1), str(6)
memory usage: 18.4 KB


## Step 3: Handle missing values

First we **identify** how many missing values each column has, then decide a strategy:
- For a numeric column like `price`, filling with the **median** is more robust to outliers
  than the mean.
- For a numeric column like `quantity`, missing usually means "we don't actually know how many
  were bought" — filling with the median quantity is a reasonable default for this exercise.
- For categorical columns like `category` and `payment_method`, we fill with a clear placeholder
  (`"Unknown"`) rather than guessing, since there's no numeric way to "average" a category.
- Rows missing critical identifying info (none in this dataset, but in general `order_id` or
  `customer_name`) would be candidates to **drop** instead of fill, since there's nothing
  meaningful to impute.


In [15]:
# Identify missing values per column
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})
missing_summary[missing_summary["missing_count"] > 0]


,missing_count,missing_pct
category,12,4.62
price,6,2.31
quantity,4,1.54
payment_method,9,3.46


In [11]:
# Fill numeric columns with the median
for col in ["price", "quantity"]:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled {col} missing values with median = {median_val}")

# Fill categorical columns with a placeholder
for col in ["category", "payment_method"]:
    if df[col].isnull().any():
        df[col] = df[col].fillna("Unknown")
        print(f"Filled {col} missing values with 'Unknown'")

# Confirm there are no missing values left
print("\nRemaining missing values:")
print(df.isnull().sum().sum())


Filled price missing values with median = 125.81
Filled quantity missing values with median = 3.0
Filled category missing values with 'Unknown'
Filled payment_method missing values with 'Unknown'

Remaining missing values:
0


## Step 4: Basic operations — filter rows, select columns

A couple of common pandas operations:
- **Filtering rows** based on a condition (e.g. orders worth more than a certain price).
- **Selecting columns** to look at a narrower slice of the data.


In [12]:
# Select a subset of columns
subset = df[["order_id", "customer_name", "product", "price", "quantity"]]
subset.head()


,order_id,customer_name,product,price,quantity
0,1103,Priya Sharma,Wireless Mouse,211.29,5.0
1,1246,Pooja Kapoor,Action Figure,120.14,5.0
2,1195,Rohan Verma,Denim Jacket,12.36,2.0
3,1118,Vikram Patel,Resistance Bands,48.49,1.0
4,1191,Karan Iyer,Shampoo,65.86,2.0


In [13]:
# Filter rows: orders where price is above the median price
price_threshold = df["price"].median()
high_value_orders = df[df["price"] > price_threshold]

print(f"Price threshold (median): {price_threshold:.2f}")
print(f"Orders above threshold: {len(high_value_orders)} out of {len(df)}")
high_value_orders.head()


Price threshold (median): 125.81
Orders above threshold: 127 out of 260


,order_id,customer_name,product,category,price,quantity,order_date,payment_method,region
0,1103,Priya Sharma,Wireless Mouse,Electronics,211.29,5.0,2024-04-12,UPI,South
8,1128,Arjun Nair,Non-stick Pan,Home & Kitchen,170.45,4.0,2024-10-09,Credit Card,West
9,1108,Sneha Singh,Dumbbell Set,Sports,133.65,3.0,2024-03-01,UPI,West
10,1028,Pooja Gupta,Non-stick Pan,Home & Kitchen,239.85,3.0,2024-10-12,Credit Card,North
11,1123,Ritu Verma,Football,Sports,165.26,3.0,2024-12-25,Credit Card,South


In [14]:
# Filter rows: combine multiple conditions (Electronics orders with quantity >= 2)
electronics_bulk = df[(df["category"] == "Electronics") & (df["quantity"] >= 2)]
electronics_bulk.head()


,order_id,customer_name,product,category,price,quantity,order_date,payment_method,region
0,1103,Priya Sharma,Wireless Mouse,Electronics,211.29,5.0,2024-04-12,UPI,South
15,1091,Arjun Gupta,Wireless Mouse,Electronics,221.62,3.0,2024-03-04,Debit Card,North
33,1059,Manoj Nair,Earbuds,Electronics,17.73,3.0,2024-02-06,Net Banking,West
36,1234,Amit Gupta,Earbuds,Electronics,206.67,3.0,2024-10-26,Credit Card,North
37,1233,Suresh Patel,Wireless Mouse,Electronics,164.80,4.0,2024-03-02,Credit Card,North


## Step 5: Remove duplicates

Duplicate rows can creep in from data entry errors, repeated exports, or merge issues. We check
for exact duplicate rows and drop them, keeping the first occurrence.


In [15]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

df = df.drop_duplicates(keep="first").reset_index(drop=True)

print(f"Shape after removing duplicates: {df.shape}")


Duplicate rows found: 10
Shape after removing duplicates: (250, 9)


## Step 6: Create a derived column

We compute `total_amount = price * quantity` — the total value of each order line.


In [16]:
PRICE_COL = "price"
QTY_COL = "quantity"

df["total_amount"] = df[PRICE_COL] * df[QTY_COL]
df[["product", PRICE_COL, QTY_COL, "total_amount"]].head()


,product,price,quantity,total_amount
0,Wireless Mouse,211.29,5.0,1056.45
1,Action Figure,120.14,5.0,600.70
2,Denim Jacket,12.36,2.0,24.72
3,Resistance Bands,48.49,1.0,48.49
4,Shampoo,65.86,2.0,131.72


In [17]:
# Quick sanity check on the new column
print("total_amount summary:")
df["total_amount"].describe()


total_amount summary:


count     250.000000
mean      413.810560
std       280.254525
min        13.860000
25%       180.120000
50%       381.055000
75%       607.110000
max      1158.250000
Name: total_amount, dtype: float64

## Step 7: Save the cleaned dataset as a new CSV file


In [18]:
CLEANED_PATH = "../data/cleaned/shopping_dataset_cleaned.csv"

df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned dataset saved to {CLEANED_PATH}")
print(f"Final shape: {df.shape}")


Cleaned dataset saved to ../data/cleaned/shopping_dataset_cleaned.csv
Final shape: (250, 10)


## Summary

- **Rows / columns before cleaning:** see Step 2 output above.
- **Missing values:** identified per column and filled — numeric columns (`price`, `quantity`)
  with the median, categorical columns (`category`, `payment_method`) with `"Unknown"`.
- **Duplicates:** exact duplicate rows were detected and removed, keeping the first occurrence.
- **Derived column:** added `total_amount = price * quantity` to represent the value of each
  order line.
- **Output:** the cleaned dataset was saved to `data/cleaned/shopping_dataset_cleaned.csv`.

This notebook demonstrates the core pandas workflow for tabular data cleaning: load → explore →
handle missing data → transform/filter → deduplicate → derive new features → export.
